In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz

def show_plotly_dashboard(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 18:00 (6 PM) and 20:00 (8 PM)
    if not (18 <= current_time_ist.hour < 20):
        print(f"Graph is currently hidden. It is only available between 6 PM and 8 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function

    # 3. Load and Clean Data
    df = pd.read_csv(csv_file_path)

    # Clean 'Installs' (Convert to numeric)
    df['Installs'] = df['Installs'].astype(str).str.replace('+', '').str.replace(',', '')
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

    # 4. Filter out categories starting with A, C, G, or S
    forbidden_starts = ('A', 'C', 'G', 'S')
    filtered_df = df[~df['Category'].str.startswith(forbidden_starts, na=False)]

    # 5. Group by Category to get total installs
    category_installs = filtered_df.groupby('Category')['Installs'].sum().reset_index()

    # 6. Get Top 5 Categories
    top_5_cats = category_installs.nlargest(5, 'Installs').copy()

    # 7. Highlight logic: Determine color based on whether installs > 1 million
    # (Since total installs for top categories usually far exceed 1M, this will likely highlight all of them,
    # but the logic scales dynamically based on your dataset)
    top_5_cats['Highlight'] = top_5_cats['Installs'].apply(
        lambda x: 'Over 1 Million' if x > 1000000 else 'Under 1 Million'
    )

    # 8. Render the Interactive Chart using Plotly
    # Note: Using a Bar chart as the dataset does not have geographical data for a Choropleth map.
    fig = px.bar(
        top_5_cats, 
        x='Category', 
        y='Installs', 
        color='Highlight',
        color_discrete_map={'Over 1 Million': '#00CC96', 'Under 1 Million': '#EF553B'},
        title='Top 5 App Categories by Global Installs (Filtered)',
        labels={'Installs': 'Total Installs', 'Category': 'App Category'}
    )
    
    fig.update_layout(xaxis_tickangle=-45)

    # To show in a standard python script:
    fig.show() 
    
    # If using Streamlit, replace fig.show() with:
    # st.plotly_chart(fig)

# Usage
show_plotly_dashboard('googleplaystore.csv')

Graph is currently hidden. It is only available between 6 PM and 8 PM IST. (Current time: 01:53 PM IST)
